# HachimiMT — Đo APP THẬT: 1 GPU vs 2 GPU (Kaggle T4×2)

Benchmark trước đo `translate_batch` THUẦN (nghìn chunk/lần) → trần 1.64×. Notebook
này đo qua **đường app thật** (`translate_text`: gồm chunking + tokenize + window +
overhead) trên cùng 1 văn bản dài, để biết multi-GPU có thật nhanh hơn TRONG APP
không (câu hỏi: window 384-chunk có thể chỉ tạo 1 sub-batch → GPU2 đói việc).

**Cách dùng:** `Settings → Accelerator → GPU T4 x2` + `Internet → on`, rồi `Run All`.
Báo lại 2 dòng `>>>` cuối.

In [ ]:
# 1. Tải code + cài thư viện
import urllib.request, zipfile, os, shutil
ZIP_URL = "https://huggingface.co/spaces/ngocdang83/HachimiMT-demo/resolve/main/hachimimt-local.zip"
urllib.request.urlretrieve(ZIP_URL, "hachimimt-local.zip")
shutil.rmtree("hachimimt", ignore_errors=True)
with zipfile.ZipFile("hachimimt-local.zip") as z: z.extractall(".")
!pip install -q -r hachimimt/requirements.txt
import ctranslate2
assert ctranslate2.get_cuda_device_count() >= 2, "Cần GPU T4 x2"
print("CT2 cuda devices:", ctranslate2.get_cuda_device_count())

In [ ]:
# 2. Dựng văn bản DÀI thật (nhiều câu đa dạng, ~vài nghìn câu = cỡ 1 chương lớn)
import os, sys, time
sys.path.insert(0, os.path.abspath("hachimimt/src"))
PARA = (
    "他抬头看向远处的山门，心中升起一股莫名的悸动。"
    "师兄说得对，我等修士当以大道为重，岂能因私情误了修行。"
    "她转身离去，没有再回头看一眼，仿佛一切已成定局。"
    "凌伊山掏出手机，查询起了最近开往雪霏市的机票。"
    "玄中门欲炼的丹药乃是凝婴丹，主材之一便是天婴果。"
    "他必须得抓紧时间了，否则一切都将来不及挽回。"
    "姑娘家世显赫，乃是百年难遇的修炼奇才，前途不可限量。"
    "公子莫要心急，此事还需从长计议，切不可操之过急。"
)
TEXT = PARA * 800        # ~6400 câu — đủ dài để GPU2 có việc nếu app feed đủ
print(f"Văn bản: {len(TEXT):,} ký tự")

In [ ]:
# 3. Đo APP THẬT: ép 1 GPU vs auto 2 GPU. Mỗi cấu hình tạo translator MỚI
#    (HACHIMIMT_GPU_INDICES đọc lúc load) → import lại sạch bằng cách reset.
import importlib, gc, time

def run_app(gpu_env, reps=3):
    # set env TRƯỚC khi (re)load module hardware/translator
    if gpu_env is None:
        os.environ.pop("HACHIMIMT_GPU_INDICES", None)
    else:
        os.environ["HACHIMIMT_GPU_INDICES"] = gpu_env
    # reload sạch để translator init lại với env mới
    for m in ["translator", "hardware"]:
        if m in sys.modules: del sys.modules[m]
    import translator as T
    from hardware import detect_hardware_profile
    tr = T.HachimiTranslator(detect_hardware_profile())
    tr.load("HachimiMT-60", T.Backend.CT2)
    n = tr.count_chunks(TEXT, "sentence")
    tr.translate_text(TEXT[:2000], chunk_mode="sentence", beam_size=1)  # warmup
    times = []
    for _ in range(reps):
        t0 = time.perf_counter()
        tr.translate_text(TEXT, chunk_mode="sentence", beam_size=1)
        times.append(time.perf_counter() - t0)
    times.sort(); med = times[len(times)//2]
    chars_s = len(TEXT) / med
    print(f"  {('2 GPU (auto)' if gpu_env is None else 'GPU='+gpu_env):16}: "
          f"{med:.1f}s · {n/med:,.0f} chunk/s · {chars_s:,.0f} chữ/s")
    del tr; gc.collect()
    return chars_s

print("Dịch APP THẬT (translate_text: chunk+tokenize+window+decode), beam=1:")
c1 = run_app("0")     # ép 1 GPU
c2 = run_app(None)    # auto = 2 GPU
print(f"\n>>> APP THẬT 2 GPU / 1 GPU: {c2/c1:.2f}×")
print(f">>> 1 GPU {c1:,.0f} chữ/s · 2 GPU {c2:,.0f} chữ/s")
print("    (so với trần benchmark 1.64× — nếu thấp hơn nhiều = window chưa feed đủ GPU2)")